In [1]:
import os
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
          'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[v] = '1'
# then numpy / xso / ssh imports BELOW this line

In [2]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys, os, time, warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sys.path.insert(0, os.path.abspath('../model'))
sys.path.insert(0, os.path.abspath('../parameter_scan'))

sys.path.insert(0, os.path.abspath('results/'))

In [5]:
print("="*98)
print("Three-way comparison: obs target vs sinusoidal model vs obs-Fourier model")
print("="*98)

OBS_KEYS = [('mcs','%7.3f'), ('mcs_med','%7.3f'), ('pico','%6.3f'), ('nano','%6.3f'),
            ('micro','%6.3f'), ('sumP','%6.3f'), ('Z200','%7.4f'), ('Z500','%8.5f'),
            ('N','%6.3f'), ('PP','%6.3f')]

for tag, alpha, rF, era_key in CHECKS:
    clim_sin = sinus[tag]
    clim_fou, _ = D['runs'][(era_key, rF)]
    obs_t = D['obs_t'][era_key]
    print(f"\n--- {tag}  |  r_F={rF}  |  era={era_key} ---")
    print(f"  {'metric':<9} {'obs_med':>8} {'obs_mn':>8}  {'sin':>8} {'Fou':>8}  "
          f"{'|Δsin|%':>9} {'|Δfou|%':>9}  closer (vs obs_med)")
    sin_wins = fou_wins = ties = 0
    for key, fmt in OBS_KEYS:
        om = obs_t['med'].get(key, np.nan)
        omn = obs_t['mean'].get(key, np.nan)
        sv = clim_sin.get(key, np.nan)
        fv = clim_fou.get(key, np.nan)
        # Distance to obs median (the Fig 4 box-plot reference)
        d_sin = abs(sv - om) / om * 100 if om else np.nan
        d_fou = abs(fv - om) / om * 100 if om else np.nan
        if not np.isfinite(d_sin) or not np.isfinite(d_fou):
            closer = '—'
        elif abs(d_sin - d_fou) < 0.5:
            closer = 'tie'; ties += 1
        elif d_sin < d_fou:
            closer = 'sin'; sin_wins += 1
        else:
            closer = 'Fou'; fou_wins += 1
        print(f"  {key:<9} {fmt%om:>8} {fmt%omn:>8}  {fmt%sv:>8} {fmt%fv:>8}  "
              f"{d_sin:>8.1f}% {d_fou:>8.1f}%   {closer}")
    print(f"  → tally: sin wins {sin_wins}  |  Fourier wins {fou_wins}  |  ties {ties}")

Three-way comparison: obs target vs sinusoidal model vs obs-Fourier model

--- post     (α=0)  |  r_F=0.1  |  era=post ---
  metric     obs_med   obs_mn       sin      Fou    |Δsin|%   |Δfou|%  closer (vs obs_med)
  mcs          2.318    4.668     3.281    3.152      41.6%     36.0%   Fou
  mcs_med        nan      nan     2.306    2.222       nan%      nan%   —
  pico         0.562    0.542     0.558    0.566       0.7%      0.7%   tie
  nano         0.272    0.267     0.248    0.244       8.9%     10.5%   sin
  micro        0.124    0.191     0.194    0.191      56.7%     54.0%   Fou
  sumP         0.459    0.581     0.468    0.468       1.9%      2.0%   tie
  Z200        0.0566   0.0660    0.0379   0.0363      33.1%     35.8%   sin
  Z500       0.03341  0.03712   0.00833  0.00829      75.1%     75.2%   tie
  N            0.959    1.235     0.852    0.768      11.2%     19.9%   sin
  PP           0.202    0.324     0.114    0.114      43.4%     43.6%   tie
  → tally: sin wins 3  |  Fo

In [3]:
import warnings, time as _t, os
import numpy as np
import xarray as xr
import seasonal_scan_harness as ssh
from xso.parscans import run_parallel_tasks
from baseline_r0_seasonal_comps import _build_fourier_func
#warnings.filterwarnings('ignore', message='Instability event')
#warnings.filterwarnings('ignore', message='solve_ivp')

# === era anchors from obs (Fourier-smoothed) ============================
forc_obs = ssh.build_forcings(['pre+recovery', 'post'])
def _smooth(monthly, n=2):
    return _build_fourier_func(monthly, period=365.0, n_harmonics=n)(np.linspace(0, 365, 365))
def _char(monthly):
    s = _smooth(monthly)
    return float(s.mean()), float((s.max()-s.min())/2), int(np.argmax(s))
POST_FN_M, POST_FN_A, POST_FN_PHI = _char(forc_obs['post']['fn'])
PRE_FN_M,  PRE_FN_A,  _           = _char(forc_obs['pre+recovery']['fn'])
POST_DE_M, POST_DE_A, _           = _char(forc_obs['post']['de'])
PRE_DE_M,  PRE_DE_A,  _           = _char(forc_obs['pre+recovery']['de'])
POST_T_M,  POST_T_A,  _           = _char(forc_obs['post']['t'])
PRE_T_M,   PRE_T_A,   _           = _char(forc_obs['pre+recovery']['t'])
PHI      = POST_FN_PHI
MID_DOY  = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])

def sinusoidal_monthly(alpha):
    """Sinusoidal F_N, d_e, T anti-phase locked to F_N peak; linear interp post (α=0) → pre+rec (α=1).
    Returns 12 mid-month values (the harness Fourier-fits these to a continuous forcing)."""
    fn_m = POST_FN_M + alpha * (PRE_FN_M - POST_FN_M)
    fn_a = POST_FN_A + alpha * (PRE_FN_A - POST_FN_A)
    de_m = POST_DE_M + alpha * (PRE_DE_M - POST_DE_M)
    de_a = POST_DE_A + alpha * (PRE_DE_A - POST_DE_A)
    t_m  = POST_T_M  + alpha * (PRE_T_M  - POST_T_M)
    t_a  = POST_T_A  + alpha * (PRE_T_A  - POST_T_A)
    cos_md = np.cos(2*np.pi*(MID_DOY - PHI)/365)
    return {
        'fn': np.maximum(fn_m + fn_a * cos_md, 0.01),
        'de': np.maximum(de_m - de_a * cos_md, 5.0),
        't':  t_m - t_a * cos_md,
    }

# === scan axes ===========================================================
ALPHAS = np.linspace(-0.5, 2.0, 50)
R_F    = np.linspace(0.0, 0.6, 50)
print(f"α:   {ALPHAS[0]:.2f}–{ALPHAS[-1]:.2f}  (Δ={ALPHAS[1]-ALPHAS[0]:.3f}, n={len(ALPHAS)})")
print(f"r_F: {R_F[0]:.3f}–{R_F[-1]:.3f}  (Δ={R_F[1]-R_F[0]:.4f}, n={len(R_F)})")
print(f"tasks: {len(ALPHAS)*len(R_F)} = 2500")
print(f"SEASONAL_SOLVER_KWARGS: {ssh.SEASONAL_SOLVER_KWARGS}\n")

# === construct (settled) + per-task kwargs ==============================
spec = ssh.allometry('maranon_ward')
GRZ  = {'KsZ': 0.23, 'sigma_log': 0.2}
IVO  = {'GrazingRouter': {'gge': 0.31}}

combos = [(a, rf) for a in ALPHAS for rf in R_F]
tasks  = [(dict(construct=spec,
                forcing=sinusoidal_monthly(a),
                fish_rate=float(rf),
                years=60, spinup=15,
                mP=0.0015, m_Z=0.1,
                grazing=GRZ, iv_overrides=IVO,
                return_traj=True),)
          for a, rf in combos]

# === result containers ==================================================
shape = (len(ALPHAS), len(R_F))
def _empty(*dims): return np.full((*shape, *dims), np.nan)

SCALAR_FIELDS = ['mcs', 'mcs_med', 'pico', 'nano', 'micro',
                 'sumP', 'sumZ', 'Z200', 'Z500', 'Z200_peak',
                 'N', 'PP', 'Export',
                 'cv_sumP', 'cv_sumZ', 'mcs_conv', 'minP', 'minZ', 'nan_frac']
CLIM_FIELDS   = ['mcs', 'pico', 'nano', 'micro', 'sumP', 'Z200', 'Z500', 'N', 'PP']
# r-dict key → storage name (r uses 'Ztot' for total zoo)
TRAJ_MAP = [('mcs','mcs'), ('pico','pico'), ('nano','nano'), ('micro','micro'),
            ('sumP','sumP'), ('Ztot','sumZ'),
            ('Z200','Z200'), ('Z500','Z500'),
            ('N','N'), ('PP','PP')]
TAIL_DAYS = 1000

scalars = {k: _empty() for k in SCALAR_FIELDS}
clims   = {k: _empty(12) for k in CLIM_FIELDS}
trajs   = {ds_name: _empty(TAIL_DAYS) for _, ds_name in TRAJ_MAP}
HASN    = np.zeros(shape, dtype=bool)

def on_result(item, done, n):
    k = item['index']; i, j = divmod(k, len(R_F))
    if not item['ok']:
        return
    clim, r = item['result']
    HASN[i, j] = bool(clim.get('has_nan', False))
    for f in SCALAR_FIELDS:
        scalars[f][i, j] = clim.get(f, np.nan)
    for f in CLIM_FIELDS:
        v = clim.get('clim_' + f)
        if v is not None and len(v) == 12:
            clims[f][i, j, :] = v
    if not HASN[i, j] and 't' in r and len(r['t']) >= TAIL_DAYS:
        for r_key, ds_name in TRAJ_MAP:
            if r_key in r:
                v = np.asarray(r[r_key])
                if len(v) == len(r['t']):
                    trajs[ds_name][i, j, :] = v[-TAIL_DAYS:]

# === run =================================================================
t0 = _t.perf_counter()
run_parallel_tasks(ssh._seasonal_worker, tasks, processes=None,
                   on_result=on_result, tally_flags=['has_nan'],
                   label='fig5_sinusoidal_50x50',
                   maxtasksperchild=1)
wall = (_t.perf_counter() - t0) / 60
print(f"\nwall: {wall:.1f} min")
print(f"trips: {int(HASN.sum())}/{HASN.size}")
print(f"|conv-1| > 0.10: {int(np.nansum(np.abs(scalars['mcs_conv'] - 1) > 0.10))}/{HASN.size}")
for k in ('mcs_med', 'micro', 'cv_sumP', 'Z200'):
    print(f"  {k:10s} range: {np.nanmin(scalars[k]):.4f} – {np.nanmax(scalars[k]):.4f}")

# === save to netCDF ======================================================
out_dir = 'results'; os.makedirs(out_dir, exist_ok=True)
nc_path = os.path.join(out_dir, 'fig5_alpha_sinusoidal_50x50_2026-06-26.nc')

ds = xr.Dataset(
    data_vars={
        **{k: (('alpha', 'rF'), scalars[k]) for k in SCALAR_FIELDS},
        **{f'clim_{k}': (('alpha', 'rF', 'month'), clims[k]) for k in CLIM_FIELDS},
        **{f'traj_{k}': (('alpha', 'rF', 'day'), trajs[k]) for _, k in TRAJ_MAP},
        'has_nan': (('alpha', 'rF'), HASN.astype(np.int8)),
    },
    coords={'alpha': ALPHAS, 'rF': R_F,
            'month': np.arange(1, 13),
            'day': np.arange(TAIL_DAYS)},
    attrs=dict(
        construct='maranon_ward (Marañón 2013 μ + Ward 2012 K_s)',
        grazing='Dutkiewicz 2020 Type III + omnivory',
        GGE=0.31, mP=0.0015, m_Z=0.1, KsZ=0.23, sigma_log=0.2,
        forcing='sinusoidal abstract — F_N, d_e, T anti-phase locked, linear interp post (α=0) ↔ pre+rec (α=1)',
        forcing_FN_mean_post=POST_FN_M, forcing_FN_mean_pre=PRE_FN_M,
        forcing_FN_amp_post=POST_FN_A,  forcing_FN_amp_pre=PRE_FN_A,
        forcing_de_mean_post=POST_DE_M, forcing_de_mean_pre=PRE_DE_M,
        forcing_de_amp_post=POST_DE_A,  forcing_de_amp_pre=PRE_DE_A,
        forcing_T_mean_post=POST_T_M,   forcing_T_mean_pre=PRE_T_M,
        forcing_T_amp_post=POST_T_A,    forcing_T_amp_pre=PRE_T_A,
        forcing_phase_day=PHI,
        years=60, spinup=15, tail_days=TAIL_DAYS,
        solver='tight RK45 (atol=1e-9, rtol=1e-6, max_step=1.0); scipy#10070 fix via maxtasksperchild=1',
        created='2026-06-26',
        notes='α=0 anchored to post obs (mean+amp); α=1 anchored to pre+rec obs (mean+amp). '
              'Era markers in (α, r_F) plane: post at (α=0, r_F=0.1); pre+rec at (α=1, r_F=0.5). '
              'See Ideas and Findings entries for the Fig 5 design context.',
    ),
)
ds.to_netcdf(nc_path)
print(f"\nsaved: {nc_path}  ({os.path.getsize(nc_path)/1e6:.1f} MB)")
print(f"reload: ds = xr.open_dataset('{nc_path}')")

α:   -0.50–2.00  (Δ=0.051, n=50)
r_F: 0.000–0.600  (Δ=0.0122, n=50)
tasks: 2500 = 2500
SEASONAL_SOLVER_KWARGS: {'method': 'RK45', 'atol': 1e-09, 'rtol': 1e-06, 'max_step': 1.0, 'instability_neg_threshold': -0.001}

--- run_parallel_tasks: 2500 tasks, 19 workers (fig5_sinusoidal_50x50) ---
     1/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 2902.4m
     2/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 1458.7m
     3/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 977.1m
     4/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 733.8m
     5/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 588.6m
     6/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 490.8m
     7/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 420.8m
     8/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 369.0m
     9/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 329.8m
    10/2500 fig5_sinusoidal_50x50 | err=0 has_nan=0 | 1.2m ETA 298.3m
    11/2